## Calculate stellar parameters

In [12]:
import sys
sys.path.append('/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject')
from functions import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown as md

In [13]:
def BailerJones():
    """
    Import distances to object from Bailer-Jones database
    Change distances to LMC and SMC
    """
    # Import
    BJ = pd.read_csv('/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/BailerJonesDistances.csv', sep=',', header=0, na_values=None)
    ID = pd.read_excel('/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/id_converter.xlsx')
    # Merge
    BJ['source_id'] = BJ["source_id"].astype(np.int64)
    ID['source_id'] = ID["source_id"].astype(np.int64)
    BJ = pd.merge(ID, BJ, on='source_id')
    BJ['source_id'] = BJ['source_id'].astype(str)

    # Change distance to LMC and SMC
    BJ.loc[BJ['id'] == 'SMC X-1', ['r_med_photogeo', 'r_med_geo']] = 62440
    BJ.loc[BJ['id'] == 'SMC X-1', ['r_lo_photogeo', 'r_lo_geo']] = 58440
    BJ.loc[BJ['id'] == 'SMC X-1', ['r_hi_photogeo', 'r_hi_geo']] = 66440

    BJ.loc[BJ['id'] == 'LMC X-1', ['r_med_photogeo', 'r_med_geo']] = 48590
    BJ.loc[BJ['id'] == 'LMC X-1', ['r_lo_photogeo', 'r_lo_geo']] = 44590
    BJ.loc[BJ['id'] == 'LMC X-1', ['r_hi_photogeo', 'r_hi_geo']] = 52590

    BJ.loc[BJ['id'] == 'LMC X-4', ['r_med_photogeo', 'r_med_geo']] = 48590
    BJ.loc[BJ['id'] == 'LMC X-4', ['r_lo_photogeo', 'r_lo_geo']] = 44590
    BJ.loc[BJ['id'] == 'LMC X-4', ['r_hi_photogeo', 'r_hi_geo']] = 52590

    return BJ

In [14]:
# Import data
df_BV = pd.read_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/B_V.xlsx")
df_falenga = pd.read_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/Falanga.xlsx")
df_stellar_params = pd.read_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/StellarParam.xlsx")
df_photometric_params = pd.read_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/PhotometricParam.xlsx")
df_BJ = BailerJones()
df_Teff = pd.read_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/Parameters/Teff.xlsx")
df_M = pd.read_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/Parameters/Mass.xlsx")
df_Lx = pd.read_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/Parameters/Lx.xlsx")

## <font color='yellow' size=5> Calculate observed luminosity from photometric filters </font>


In [15]:
df_L = {
    'id': [],
    'Av': [],
    'Av_err': [],
    'E(B-V)': [],
    'E(B-V)_err': [],
    '(B-V)': [],
    '(B-V)_err': [],
    'V': [],
    'V_err': [],
    'BCv': [],
    '(B-V)0': [],
    'Rv': [],
    'Rv_err': [],
    'Mbol': [],
    'Mv': [],
    'id': [],
    'ST': [],
    'L': [],
    'L+': [],
    'L-': [],
    'd': [],
    'd+': [],
    'd-': [],
    'Teff': [], 
    'Teff_err': []
    }

df_hmxb =  pd.merge(df_BV, df_falenga, on='id')
df_hmxb = pd.merge(df_hmxb, df_Teff, on='id')
df_hmxb = pd.merge(df_hmxb, df_BJ, on='id')
df_hmxb = df_hmxb[['id', 'ST', 'Teff', 
                   'r_med_photogeo', 'r_lo_photogeo', 'r_hi_photogeo', 
                   'B-V', 'B-V_err', 'V', 'V_err']]


for index, row in df_hmxb.iterrows():
    # Object id
    id = row['id']

    # Spectral type of object
    spectral_type = row['ST']

    # Effective temperature based on spectral type
    Teff = row['Teff']
    Teff_err = 1000

    # Distance
    distance = row['r_med_photogeo']
    distance_low = row['r_lo_photogeo']
    distance_high = row['r_hi_photogeo']
    Sigmad_low = distance - distance_low
    Sigmad_high = distance_high - distance

    # Expected (B-V)0 of object based on spectral type
    BV0 = interpolate(df2=df_photometric_params, spectral_type=spectral_type, quantity='(B-V)0')
    BV0_err = 0

    # Observed (B-V) of object based on simbad filters
    BVobs = row['B-V']
    BVobs_err = row['B-V_err']

    # Bolometric correction (BC)
    if spectral_type[0] == 'B':
        if id == 'Vela X-1':
            BCv = -2.16
        elif id == 'SMC X-1':
            BCv = -2.35
        elif id == '4U1538-52':
            BCv = -2.32
    elif spectral_type[0] == 'O':
        BCv = interpolate(df2=df_photometric_params, spectral_type=spectral_type, quantity='BCv')

    # Calculate extinction
    if 'LMC' in id:
        Rv, Rv_err = 3.40, 0
        Av, Av_err = extinction_and_error(3.40, 0, BVobs, BVobs_err, BV0, BV0_err)
    elif 'SMC' in id:
        Rv, Rv_err = 2.53, 0
        Av, Av_err = extinction_and_error(2.53, 0, BVobs, BVobs_err, BV0, BV0_err)
    else:
        Rv, Rv_err = 3.16, 0.15
        Av, Av_err = extinction_and_error(3.2, 0, BVobs, BVobs_err, BV0, BV0_err)
    # Calculate E(B-V)
    EBV = BVobs - BV0
    EBV_err = np.sqrt(BVobs_err**2 + BV0_err**2)


    # Visual magnitude
    mv = row['V']

    # Calculate Absulute magnitude (visual)
    Mv = mv - 5 * np.log10(distance) + 5 - Av

    # Calculate bolomatric absolute magnitude
    Mbol = Mv + BCv

    # Calculate the luminosity in solar luminosities
    L = 10**((Mbol - 4.74) / (-2.5))

    # Calculate the error on the luminosity
    L_err_low, L_err_high = luminosity_error_asymmetric(BCv, 0.1, mv, row['V_err'], distance, Sigmad_high, Sigmad_low, Av, Av_err)

    # Save the other parameters
    df_L['id'].append(id)
    df_L['ST'].append(spectral_type)
    df_L['Av'].append(Av)
    df_L['Av_err'].append(Av_err)
    df_L['V'].append(mv)
    df_L['V_err'].append(row['V_err'])
    df_L['(B-V)'].append(BVobs)
    df_L['(B-V)_err'].append(BVobs_err)
    df_L['(B-V)0'].append(BV0)
    df_L['BCv'].append(BCv)
    df_L['E(B-V)'].append(EBV)
    df_L['E(B-V)_err'].append(EBV_err)
    df_L['Rv'].append(Rv)
    df_L['Rv_err'].append(Rv_err)
    df_L['Mbol'].append(Mbol)
    df_L['Mv'].append(Mv)

    df_L['L'].append(L)
    df_L['L+'].append(L_err_high)
    df_L['L-'].append(L_err_low)
    df_L['d'].append(distance)
    df_L['d+'].append(Sigmad_high)
    df_L['d-'].append(Sigmad_low)
    df_L['Teff'].append(Teff)
    df_L['Teff_err'].append(Teff_err)

df_L = pd.DataFrame(df_L)

In [16]:
df_L.to_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/Parameters/Luminosity.xlsx")

## <font color='yellow' size=5>Radius</font>

- Ropt: Falanga
- Rst: Martins
- Reff: L=...

In [17]:
df_R = {
    "id": [],
    "ST": [],
    "Reff": [],
    "Reff+": [],
    "Reff-": [],
    "Ropt": [],
    "Ropt_err": [],
    "Rst": [],
    "Rst_err": []
}

df_hmxb =  pd.merge(df_L[['id', 'L', 'L+', 'L-', 'Teff', 'Teff_err']], df_falenga, on='id')


for index, row  in df_hmxb.iterrows():
    # id
    id = row['id']

    # Spectral type
    spectral_type = row['ST']

    # True luminosity
    L = row['L']
    L_err_high = row['L+']
    L_err_low = row['L-']
    
    # Effective temperature from model
    Teff = row['Teff']
    Teff_err = row['Teff_err']

    # Calculate the radius
    R, R_err_high, R_err_low = expected_radius_error_asymmetric(L, L_err_high, L_err_low, Teff, Teff_err)
    # Radius based on spectral type
    R_ST = interpolate(df_stellar_params, spectral_type, 'R')

    df_R['Reff'].append(R)
    df_R['Reff+'].append(R_err_high)
    df_R['Reff-'].append(R_err_low)
    df_R['id'].append(id)
    df_R['ST'].append(spectral_type)
    df_R['Ropt'].append(row['Ropt'])
    df_R['Ropt_err'].append(row['Ropt_error'])
    df_R['Rst'].append(R_ST)
    df_R['Rst_err'].append(0.15 * R_ST)

df_R = pd.DataFrame(df_R)

In [18]:
df_R.to_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/Parameters/Radius.xlsx")

## <font color='yellow' size=5>Roche Lobe radius</font>

In [19]:
df_RL = {
    "id": [],
    "RL": [],
    "RL_err": [],
    "RLq": [],
    "RLq_err": [],
    "RLg": [],
    "RLg_err": []
}

for index, row in df_falenga.iterrows():
    id = row['id']

    # Orbital separation
    a, a_err = row['a'], row['a_error']

    # Masses of optical and x-ray sources
    Mopt, Mopt_err = row['Mopt'], row['Mopt_error']
    Mns, Mns_err =  row['Mns'], row['Mns_error']

    # Radius of optical source
    R, R_err = row['Ropt'], row['Ropt_error']

    # Calculate roche-lobe radius in 3 ways
    RLfal, RLfal_err = row['RL/a'] * row['a'], np.sqrt((row['a'] * row['RL/a_error']) ** 2 + (row['RL/a'] * row['a_error']) ** 2)
    RLq, RLq_err = roche_lobe_radius_q(a, a_err, Mopt, Mopt_err, Mns, Mns_err)
    RLg, RLg_err = roche_lobe_radius_g(a, a_err, Mopt, Mopt_err, R, R_err)

    df_RL["id"].append(id)
    df_RL["RL"].append(RLfal)
    df_RL["RL_err"].append(RLfal_err)
    df_RL["RLq"].append(RLq)
    df_RL["RLq_err"].append(RLq_err)
    df_RL["RLg"].append(RLg)
    df_RL["RLg_err"].append(RLg_err)

df_RL = pd.DataFrame(df_RL)

In [20]:
df_RL.to_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/Parameters/RocheLobeRadius.xlsx")

## <font color='yellow' size=5>All Parameters</font>

In [21]:
def replace_error_in_columns(df):
    # Rename columns by replacing '_error' with '_err'
    df.rename(columns=lambda col: col.replace('_error', '_err'), inplace=True)
    return df

In [22]:
df_AllParam = pd.merge(df_L, df_R.drop(columns=['ST']), on='id')
df_AllParam = pd.merge(df_AllParam, df_RL, on='id')
df_AllParam = pd.merge(df_AllParam, df_falenga.drop(columns=['Type', 'ST', 'Ropt', 'Ropt_error']), on='id')
df_AllParam = pd.merge(df_AllParam, df_M[['id', 'Mhrd', 'Mhrd_err']], on='id')
df_AllParam = pd.merge(df_AllParam, df_Lx, on='id')

df_AllParam = replace_error_in_columns(df_AllParam)
df_AllParam.to_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/Parameters/AllParam.xlsx")